# cuda setup smoke checks

Use this notebook to inspect and run the bounded CUDA setup checks. Select lmlab's `.venv` Python kernel first. Edit the CUDA program in [`src/gpu_primitives/smoke_cuda.cu`](../src/gpu_primitives/smoke_cuda.cu); the notebook compiles that file directly rather than duplicating its source.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src').is_dir():
            return candidate
    raise RuntimeError(f'could not find the lmlab root from {start}')


repo_root = find_repo_root(Path.cwd().resolve())
cuda_home = Path(os.environ.get('CUDA_HOME', '/usr/local/cuda-13.3'))
cuda_arch = os.environ.get('CUDA_ARCH', 'sm_86')
cuda_source = repo_root / 'src/gpu_primitives/smoke_cuda.cu'
build_dir = repo_root / '.build'
cuda_binary = build_dir / 'smoke_cuda'
triton_smoke = repo_root / 'src/gpu_primitives/smoke_triton.py'

print(f'repository: {repo_root}')
print(f'cuda source: {cuda_source.relative_to(repo_root)}')
print(f'cuda toolkit: {cuda_home}')

## compile the cuda c++ smoke

In [ ]:
build_dir.mkdir(exist_ok=True)
subprocess.run(
    [
        str(cuda_home / 'bin/nvcc'),
        f'-arch={cuda_arch}',
        '-std=c++17',
        str(cuda_source),
        '-o',
        str(cuda_binary),
    ],
    check=True,
    text=True,
)
print(f'compiled {cuda_binary.relative_to(repo_root)}')

## run the cuda c++ smoke

In [ ]:
subprocess.run([str(cuda_binary)], check=True, text=True)

## check memory access with compute sanitizer

In [ ]:
subprocess.run(
    [
        str(cuda_home / 'bin/compute-sanitizer'),
        '--tool',
        'memcheck',
        '--error-exitcode',
        '1',
        str(cuda_binary),
    ],
    check=True,
    text=True,
)

## optional triton smoke

This uses the notebook's Python environment so its PyTorch and Triton packages stay aligned.

In [ ]:
subprocess.run([sys.executable, str(triton_smoke)], check=True, text=True)

## observations

Record the machine, toolchain versions, observed output, and any follow-up question here after running the checks.